# Qwen3-1.7B — SRD-4 · ARM · Axiom MET

| Output | Size (est.) | Description |
|---|---|---|
| `qwen3_1b7_srd.axm` | ~1.1 GB | Signed SRD-4 container (~4.5 bpw) |
| `qwen3_1b7_q4km.gguf` | ~1.1 GB | GGUF Q4_K_M — runs in llama.cpp / PocketPal |
| `qwen3_1b7_q4km.axiom_meta.json` | ~5 KB | MET slot map + hydration policy sidecar |

**Rival:** `bartowski/gemma-3-1b-it-GGUF` Q4_K_M — benchmarked in Cell 8.

**ARM build:** cmake auto-detects Metal (Apple Silicon), CUDA ARM (Jetson), ARM NEON (CPU), or x86 CUDA (Colab).

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Platform detect · clone axiom · cmake llama.cpp (ARM-aware)  (~5–15 min)
# ══════════════════════════════════════════════════════════════════════════════
import os, platform, secrets, subprocess, sys, time
from pathlib import Path

SYSTEM = platform.system()   # 'Darwin' | 'Linux'
ARCH   = platform.machine()  # 'arm64' | 'aarch64' | 'x86_64'

# ── Workspace (auto: /content on Colab, ~/axiom_workspace elsewhere) ──────────
if SYSTEM == 'Darwin':
    WORKSPACE = Path.home() / 'axiom_workspace'
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.home() / 'axiom_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)

REPO      = WORKSPACE / 'axiom'
LLAMA_DIR = WORKSPACE / 'llama.cpp'
OUT_DIR   = WORKSPACE / 'qwen3_srd'
OUT_DIR.mkdir(parents=True, exist_ok=True)
KEY_FILE  = WORKSPACE / 'axiom_master.key'

REPO_URL    = 'https://github.com/orivael-dev/axiom.git'
REPO_BRANCH = 'claude/srd-prototype-benchmark-JRtv1'

# ── Detect build backend ───────────────────────────────────────────────────────
def _cuda_arch():
    try:
        r = subprocess.run(
            ['python3', '-c',
             'import torch; p=torch.cuda.get_device_properties(0); '
             'print(p.major*10+p.minor)'],
            capture_output=True, text=True)
        return int(r.stdout.strip())
    except Exception:
        return None

IS_MACOS_ARM = (SYSTEM == 'Darwin' and ARCH == 'arm64')
IS_LINUX_ARM = (SYSTEM == 'Linux' and ARCH in ('aarch64', 'arm64'))
HAS_CUDA     = subprocess.run(['which', 'nvcc'], capture_output=True).returncode == 0
CUDA_ARCH    = _cuda_arch() if HAS_CUDA else None

if IS_MACOS_ARM:
    CMAKE_EXTRA   = ['-DGGML_METAL=ON', '-DGGML_BLAS=OFF']
    N_GPU_LAYERS  = 99
    BACKEND_LABEL = 'Metal (Apple Silicon)'
elif IS_LINUX_ARM and HAS_CUDA:
    arch = CUDA_ARCH or 87
    CMAKE_EXTRA   = ['-DGGML_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={arch}']
    N_GPU_LAYERS  = 99
    BACKEND_LABEL = f'CUDA ARM (arch={arch})'
elif IS_LINUX_ARM:
    CMAKE_EXTRA   = []          # NEON auto-detected on ARM Linux
    N_GPU_LAYERS  = 0
    BACKEND_LABEL = 'ARM NEON (CPU)'
elif HAS_CUDA:
    arch = CUDA_ARCH or 75
    CMAKE_EXTRA   = ['-DGGML_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={arch}']
    N_GPU_LAYERS  = 99
    BACKEND_LABEL = f'CUDA x86 (arch={arch})'
else:
    CMAKE_EXTRA   = []
    N_GPU_LAYERS  = 0
    BACKEND_LABEL = 'CPU only'

print(f'Platform    : {SYSTEM} {ARCH}')
print(f'Build target: {BACKEND_LABEL}')
print(f'Workspace   : {WORKSPACE}')

# ── pip install ────────────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'accelerate', 'psutil',
                'huggingface_hub', 'sentencepiece', 'safetensors', 'tqdm'],
               check=True)
print('✓ pip packages installed')

# ── Clone axiom ─────────────────────────────────────────────────────────────
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                    REPO_URL, str(REPO)], check=True)
    print(f'✓ axiom cloned  ({REPO_BRANCH})')
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'],
                   capture_output=True)
    print(f'✓ axiom present  ({REPO_BRANCH})')
sys.path.insert(0, str(REPO))

# ── Persist AXIOM_MASTER_KEY ───────────────────────────────────────────────────
if KEY_FILE.is_file() and not os.environ.get('AXIOM_MASTER_KEY'):
    os.environ['AXIOM_MASTER_KEY'] = KEY_FILE.read_text().strip()
    print('AXIOM_MASTER_KEY restored from key file')
elif not os.environ.get('AXIOM_MASTER_KEY'):
    key = secrets.token_hex(32)
    os.environ['AXIOM_MASTER_KEY'] = key
    KEY_FILE.write_text(key)
    print(f'AXIOM_MASTER_KEY generated → {KEY_FILE}')
    print('  ⚠ back this up — required to verify the .axm later')
else:
    print('AXIOM_MASTER_KEY already set')

# ── Clone llama.cpp ────────────────────────────────────────────────────────────
if not LLAMA_DIR.is_dir():
    print('Cloning llama.cpp ...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp',
                    str(LLAMA_DIR)], check=True)
    print('✓ llama.cpp cloned')
else:
    print('✓ llama.cpp present')

# ── cmake build ────────────────────────────────────────────────────────────────
Q_BIN     = LLAMA_DIR / 'build/bin/llama-quantize'
CLI_BIN   = LLAMA_DIR / 'build/bin/llama-cli'
BENCH_BIN = LLAMA_DIR / 'build/bin/llama-bench'

if not Q_BIN.is_file():
    print(f'\nBuilding llama.cpp ({BACKEND_LABEL}) — ~5–15 min ...')
    t0 = time.time()
    subprocess.run(
        ['cmake', '-B', 'build', '-DCMAKE_BUILD_TYPE=Release'] + CMAKE_EXTRA,
        cwd=LLAMA_DIR, check=True,
    )
    subprocess.run(
        ['cmake', '--build', 'build',
         '--target', 'llama-quantize', 'llama-cli', 'llama-bench',
         f'-j{os.cpu_count() or 4}'],
        cwd=LLAMA_DIR, check=True,
    )
    print(f'✓ llama.cpp built in {(time.time()-t0)/60:.1f} min')
else:
    print('✓ llama.cpp already built')

for lbl, p in [('llama-quantize', Q_BIN), ('llama-cli', CLI_BIN), ('llama-bench', BENCH_BIN)]:
    print(f'  {lbl:<18} {"✓" if p.is_file() else "⚠ missing"}  {p}')

print(f'\n✓ Cell 1 complete — backend: {BACKEND_LABEL}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download Qwen3-1.7B directly from HuggingFace  (~5–10 min, ~3.4 GB)
#
# No gated model — downloads directly from:  https://huggingface.co/Qwen/Qwen3-1.7B
# ══════════════════════════════════════════════════════════════════════════════
import json
from huggingface_hub import snapshot_download
from pathlib import Path

try:
    OUT_DIR
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    OUT_DIR   = WORKSPACE / 'qwen3_srd'
    OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID  = 'Qwen/Qwen3-1.7B'
MODEL_DIR = OUT_DIR / 'qwen3_1b7'

if not MODEL_DIR.is_dir():
    print(f'Downloading {MODEL_ID} from HuggingFace ...')
    snapshot_download(
        MODEL_ID,
        local_dir=str(MODEL_DIR),
        ignore_patterns=['*.bin'],   # prefer safetensors
    )
    print(f'✓ Downloaded → {MODEL_DIR}')
else:
    print(f'✓ Already present: {MODEL_DIR}')

# ── Print architecture ─────────────────────────────────────────────────────────
cfg_path = MODEL_DIR / 'config.json'
if cfg_path.is_file():
    cfg      = json.loads(cfg_path.read_text())
    HIDDEN   = cfg.get('hidden_size', 2048)
    LAYERS   = cfg.get('num_hidden_layers', 28)
    VOCAB    = cfg.get('vocab_size', 151936)
    HEADS    = cfg.get('num_attention_heads', 16)
    KV_HEADS = cfg.get('num_key_value_heads', HEADS)
    print(f'\n  Architecture : {LAYERS} layers · hidden {HIDDEN} · vocab {VOCAB:,}')
    print(f'  Attention    : {HEADS} heads · {KV_HEADS} KV heads (GQA)')
    fp16_gb = HIDDEN * HIDDEN * 12 * LAYERS * 2 / 1024**3
    print(f'  Est. FP16    : {fp16_gb:.1f} GB  |  Q4_K_M: ~{fp16_gb*0.30:.1f} GB')
else:
    HIDDEN, LAYERS, VOCAB, HEADS, KV_HEADS = 2048, 28, 151936, 16, 8
    print('  [warn] config.json not found — using Qwen3-1.7B defaults')

print('\n✓ Cell 2 complete — proceed to Cell 3')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — SRD-4 pack: Qwen3-1.7B → qwen3_1b7_srd.axm  (~10–20 min)
#
# SRD-4 = W4 base quantization (α=0), pure 4-bit, ~4.5 bpw.
# Signs every weight shard with HMAC-SHA256.
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys, time
from pathlib import Path

try:
    WORKSPACE
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    REPO      = WORKSPACE / 'axiom'
    OUT_DIR   = WORKSPACE / 'qwen3_srd'
    MODEL_DIR = OUT_DIR / 'qwen3_1b7'
    sys.path.insert(0, str(REPO))

AXM_PATH    = OUT_DIR / 'qwen3_1b7_srd.axm'
STATS_JSON  = OUT_DIR / 'qwen3_1b7_pack_stats.json'
PACK_SCRIPT = REPO / 'research' / 'quant' / 'pack_to_axm.py'

assert MODEL_DIR.is_dir(),   f'Run Cell 2 first — {MODEL_DIR} not found'
assert PACK_SCRIPT.is_file(), 'pack_to_axm.py not found — run Cell 1 first'

if AXM_PATH.is_file():
    print(f'✓ AXM already exists: {AXM_PATH}')
else:
    print('Packing Qwen3-1.7B with SRD-4 (W4, α=0, ~4.5 bpw) ...')
    t0 = time.time()
    r = subprocess.run([
        sys.executable, str(PACK_SCRIPT),
        '--model',      str(MODEL_DIR),   # local path — no re-download
        '--srd4',                          # W4-only, no D8 residue → ~4.5 bpw
        '--group-size', '64',
        '--output',     str(AXM_PATH),
        '--stats-json', str(STATS_JSON),
    ], cwd=str(REPO))
    elapsed = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError(f'SRD pack failed (rc={r.returncode})')
    gb = AXM_PATH.stat().st_size / 1024**3
    print(f'✓ {AXM_PATH.name}  ({gb:.2f} GB)  {elapsed/60:.1f} min')

FINGERPRINT = ''
if STATS_JSON.is_file():
    s = json.loads(STATS_JSON.read_text())
    FINGERPRINT = s.get('fingerprint', '')
    print(f'  fingerprint : {FINGERPRINT}')
    print(f'  bpw         : {s.get("bpw_theoretical")}')
    print(f'  proofs      : {s.get("proofs")}')

print('\n✓ Cell 3 complete — proceed to Cell 4')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Verify .axm proof ledger  (~5–15 s)
# ══════════════════════════════════════════════════════════════════════════════
import json, subprocess, sys
from pathlib import Path

try:
    WORKSPACE
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    REPO    = WORKSPACE / 'axiom'
    OUT_DIR = WORKSPACE / 'qwen3_srd'
    sys.path.insert(0, str(REPO))

AXM_PATH = OUT_DIR / 'qwen3_1b7_srd.axm'
assert AXM_PATH.is_file(), 'qwen3_1b7_srd.axm not found — run Cell 3 first'

result = subprocess.run(
    [sys.executable, str(REPO / 'axm_cli.py'), 'verify', str(AXM_PATH)],
    cwd=str(REPO), capture_output=True, text=True,
)
try:
    data = json.loads(result.stdout)
except json.JSONDecodeError:
    data = {'verified': False,
            'error': result.stdout[-400:] + result.stderr[-200:]}

ok     = data.get('verified', False)
fp     = data.get('fingerprint', '?')
proofs = data.get('proofs_checked', '?')

print(f'{"✓ VERIFIED" if ok else "✗ FAILED"}')
print(f'  fingerprint    : {fp}')
print(f'  proofs checked : {proofs}')

if not ok:
    print(f'  error: {data.get("error")}')
    raise RuntimeError('Verification failed — .axm may have been tampered with')

FINGERPRINT = fp
print('\n✓ Cell 4 complete — proceed to Cell 5')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Extract .axm → GGUF Q4_K_M  (~10–20 min)
#
# .axm (SRD-4 W4)  →  reconstruct FP16  →  convert_hf_to_gguf.py
#   →  model_f16.gguf  →  llama-quantize Q4_K_M  →  qwen3_1b7_q4km.gguf
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

try:
    WORKSPACE
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE   = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    REPO        = WORKSPACE / 'axiom'
    LLAMA_DIR   = WORKSPACE / 'llama.cpp'
    OUT_DIR     = WORKSPACE / 'qwen3_srd'
    sys.path.insert(0, str(REPO))

AXM_PATH    = OUT_DIR / 'qwen3_1b7_srd.axm'
GGUF_PATH   = OUT_DIR / 'qwen3_1b7_q4km.gguf'
AXM_TO_GGUF = REPO / 'research' / 'quant' / 'axm_to_gguf.py'

assert AXM_PATH.is_file(),  'Run Cell 3 first'
assert (LLAMA_DIR / 'build/bin/llama-quantize').is_file(), 'Run Cell 1 first'

if GGUF_PATH.is_file():
    print(f'✓ GGUF already exists: {GGUF_PATH}')
else:
    print('Extracting AXM → GGUF Q4_K_M ...')
    t0 = time.time()
    r = subprocess.run([
        sys.executable, str(AXM_TO_GGUF),
        '--container', str(AXM_PATH),
        '--gguf-out',  str(GGUF_PATH),
        '--llamacpp',  str(LLAMA_DIR),
        '--quant',     'Q4_K_M',
    ], cwd=str(REPO))
    elapsed = time.time() - t0
    if r.returncode != 0:
        raise RuntimeError('GGUF extraction failed')
    gb = GGUF_PATH.stat().st_size / 1024**3
    print(f'✓ {GGUF_PATH.name}  ({gb:.2f} GB)  {elapsed/60:.1f} min')

print(f'  SRD GGUF: {GGUF_PATH.stat().st_size/1024**2:.0f} MB')
print('\n✓ Cell 5 complete — proceed to Cell 6')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Download rival: Gemma 3 1B Q4_K_M GGUF  (~3–5 min, ~600 MB)
#
# Downloads the pre-quantized GGUF from bartowski/gemma-3-1b-it-GGUF.
# No conversion step — this is the benchmark rival used in Cell 8.
# ══════════════════════════════════════════════════════════════════════════════
from huggingface_hub import hf_hub_download, list_repo_files
from pathlib import Path

try:
    OUT_DIR
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    OUT_DIR   = WORKSPACE / 'qwen3_srd'
    OUT_DIR.mkdir(parents=True, exist_ok=True)

RIVAL_REPO = 'bartowski/gemma-3-1b-it-GGUF'

# Auto-discover the Q4_K_M file
print(f'Listing {RIVAL_REPO} ...')
try:
    _all = list(list_repo_files(RIVAL_REPO))
    files = [f for f in _all if 'Q4_K_M' in f and f.endswith('.gguf')]
except Exception as e:
    print(f'  list_repo_files error ({e}) — using default filename')
    files = ['gemma-3-1b-it-Q4_K_M.gguf']

if not files:
    raise RuntimeError(f'No Q4_K_M GGUF found in {RIVAL_REPO}')

RIVAL_FILENAME = sorted(files)[0]
RIVAL_GGUF     = OUT_DIR / RIVAL_FILENAME

if RIVAL_GGUF.is_file():
    print(f'✓ Rival already present: {RIVAL_GGUF.name}')
else:
    print(f'Downloading {RIVAL_FILENAME} ...')
    path = hf_hub_download(
        RIVAL_REPO, RIVAL_FILENAME,
        local_dir=str(OUT_DIR),
        local_dir_use_symlinks=False,
    )
    RIVAL_GGUF = Path(path)
    print(f'✓ {RIVAL_GGUF.name}  ({RIVAL_GGUF.stat().st_size/1024**2:.0f} MB)')

print(f'\n  Rival  : Gemma 3 1B IT Q4_K_M  ({RIVAL_GGUF.stat().st_size/1024**2:.0f} MB)')
print(f'  Target : Qwen3-1.7B SRD Q4_K_M ({(OUT_DIR/"qwen3_1b7_q4km.gguf").stat().st_size/1024**2:.0f} MB)')
print('\n✓ Cell 6 complete — proceed to Cell 7')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Axiom MET sidecar for Qwen3-1.7B
#
# Reads config.json, writes qwen3_1b7_q4km.axiom_meta.json:
#   - embedding slot (F16, always pinned)
#   - transformer chunks: early / factual / reasoning / governance
#   - hydration policy per Axiom intent class
#   - per-intent RAM + UFS load time estimates
#   - .axm fingerprint embedded
# ══════════════════════════════════════════════════════════════════════════════
import json, time
from pathlib import Path

try:
    OUT_DIR
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    OUT_DIR   = WORKSPACE / 'qwen3_srd'
    FINGERPRINT = ''

try:
    _fp = FINGERPRINT
except NameError:
    _fp = ''

MODEL_DIR  = OUT_DIR / 'qwen3_1b7'
GGUF_PATH  = OUT_DIR / 'qwen3_1b7_q4km.gguf'
assert MODEL_DIR.is_dir(), 'Run Cell 2 first'
assert GGUF_PATH.is_file(), 'Run Cell 5 first'

# ── Architecture from config.json ─────────────────────────────────────────────
cfg      = json.loads((MODEL_DIR / 'config.json').read_text())
HIDDEN   = cfg.get('hidden_size', 2048)
LAYERS   = cfg.get('num_hidden_layers', 28)
VOCAB    = cfg.get('vocab_size', 151936)
HEADS    = cfg.get('num_attention_heads', 16)
KV_HEADS = cfg.get('num_key_value_heads', HEADS)

# ── Embedding slot (F16, always pinned) ───────────────────────────────────────
EMBEDDING_MB = round(VOCAB * HIDDEN * 2 / 1024**2, 1)   # F16

# ── Transformer chunk boundaries ──────────────────────────────────────────────
# Proportions from Axiom spec: early 20% / factual 20% / reasoning 37% / governance 23%
def _split_layers(n, fracs):
    cuts, lo = [], 0
    for f in fracs[:-1]:
        hi = lo + max(1, round(n * f))
        cuts.append((lo, min(hi, n) - 1))
        lo = min(hi, n)
    cuts.append((lo, n - 1))
    return cuts

SLOT_NAMES = ['early', 'factual', 'reasoning', 'governance']
FRACS      = [0.20, 0.20, 0.37, 0.23]
RANGES     = _split_layers(LAYERS, FRACS)

# Q4_K_M ~4.85 bpw; non-embedding params ≈ 12 * hidden^2 * layers
XFMR_PARAMS = 12 * HIDDEN**2 * LAYERS

SLOT_RANGES = {}
for name, (lo, hi) in zip(SLOT_NAMES, RANGES):
    n_layer = hi - lo + 1
    frac    = n_layer / LAYERS
    mb      = round(XFMR_PARAMS * frac * 4.85 / 8 / 1024**2)
    SLOT_RANGES[name] = {
        'layers': f'{lo}-{hi}', 'first_layer': lo, 'last_layer': hi,
        'mb': mb, 'precision': 'Q4_K_M',
    }

HYDRATION_POLICY = {
    'INFORM':    ['early'],
    'CLARIFY':   ['early', 'governance'],
    'REFUSE':    ['early', 'governance'],
    'UNCERTAIN': ['early', 'governance'],
    'HARM':      ['early', 'factual', 'reasoning', 'governance'],
    'DECEIVE':   ['early', 'factual', 'reasoning', 'governance'],
}

STORAGE_MBS = 1500      # UFS 3.1 reference (MB/s)
slot_mb     = {s: SLOT_RANGES[s]['mb'] for s in SLOT_NAMES}

intent_ram = {}
for intent, chunks in HYDRATION_POLICY.items():
    xfmr_mb  = sum(slot_mb[c] for c in chunks)
    total_mb = EMBEDDING_MB + xfmr_mb
    load_ms  = round(xfmr_mb / STORAGE_MBS * 1000, 1)
    intent_ram[intent] = {
        'chunks': chunks, 'transformer_mb': xfmr_mb,
        'total_mb': total_mb, 'ufs_load_ms': load_ms,
    }

chunk_map = {}
for slot, info in SLOT_RANGES.items():
    for idx in range(info['first_layer'], info['last_layer'] + 1):
        chunk_map[str(idx)] = slot

meta = {
    'axiom_version':   '1.4',
    'generated_at':    time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model_id':        'Qwen/Qwen3-1.7B',
    'gguf_path':       str(GGUF_PATH),
    'gguf_mb':         round(GGUF_PATH.stat().st_size / 1024**2),
    'fingerprint':     _fp,
    'architecture':    {'hidden_size': HIDDEN, 'num_layers': LAYERS,
                        'vocab_size': VOCAB, 'num_attention_heads': HEADS,
                        'num_kv_heads': KV_HEADS},
    'embedding_slot':  {
        'chunk': 'embedding', 'mb': EMBEDDING_MB, 'precision': 'F16',
        'always_pinned': True,
        'note': f'{VOCAB:,}-token vocabulary — {EMBEDDING_MB:.0f} MB F16',
    },
    'transformer_chunks': SLOT_RANGES,
    'chunk_map':          chunk_map,
    'hydration_policy':   HYDRATION_POLICY,
    'intent_ram_budget':  intent_ram,
    'storage_speed_mbs':  STORAGE_MBS,
    'between_met_floor_mb': EMBEDDING_MB,
    'peak_harm_mb':       EMBEDDING_MB + sum(slot_mb.values()),
}

sidecar_path = GGUF_PATH.with_suffix('.axiom_meta.json')
sidecar_path.write_text(json.dumps(meta, indent=2) + '\n')
print(f'✓ MET sidecar → {sidecar_path}')

print()
print('MET SLOT SUMMARY  (Qwen3-1.7B)')
print(f'  Architecture : {LAYERS} layers · hidden {HIDDEN} · vocab {VOCAB:,}  '
      f'(GQA {HEADS}h→{KV_HEADS}kv)')
print()
print(f'  {"Slot":<14}  {"Layers":<10}  {"MB":>6}  {"Prec":>8}  Always?')
print('  ' + '─' * 52)
print(f'  {"embedding":<14}  {"—embed—":<10}  {EMBEDDING_MB:>6.0f}  {"F16":>8}  ✓ pinned')
for slot, info in SLOT_RANGES.items():
    print(f'  {slot:<14}  {info["layers"]:<10}  {info["mb"]:>6}  {"Q4_K_M":>8}  on demand')

print()
print('HYDRATION POLICY  (intent → transformer chunks)')
print(f'  {"Intent":<10}  {"Chunks":<36}  {"Total MB":>9}  {"UFS ms":>7}')
print('  ' + '─' * 68)
for intent, info in intent_ram.items():
    print(f'  {intent:<10}  {"+".join(info["chunks"]):<36}  {info["total_mb"]:>9}  {info["ufs_load_ms"]:>5.1f}ms')

if _fp:
    print(f'\n  Fingerprint : {_fp}')
print('\n✓ Cell 7 complete — proceed to Cell 8')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — ARM inference benchmark: Qwen3-1.7B SRD vs Gemma 3 1B Q4_K_M
#
# Uses llama-bench (PP=512, TG=128, r=3) when available; falls back to
# llama-cli timing lines. Prints side-by-side comparison table.
# ══════════════════════════════════════════════════════════════════════════════
import re, subprocess, time
from pathlib import Path

try:
    WORKSPACE
except NameError:
    import platform as _p
    _s, _a = _p.system(), _p.machine()
    WORKSPACE   = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    LLAMA_DIR   = WORKSPACE / 'llama.cpp'
    OUT_DIR     = WORKSPACE / 'qwen3_srd'
    IS_MACOS_ARM = (_s == 'Darwin' and _a == 'arm64')
    IS_LINUX_ARM = (_s == 'Linux'  and _a in ('aarch64', 'arm64'))
    HAS_CUDA     = subprocess.run(['which', 'nvcc'], capture_output=True).returncode == 0
    N_GPU_LAYERS = 99 if (IS_MACOS_ARM or HAS_CUDA) else 0
    BACKEND_LABEL = ('Metal' if IS_MACOS_ARM else
                     'CUDA ARM' if (IS_LINUX_ARM and HAS_CUDA) else
                     'ARM NEON' if IS_LINUX_ARM else
                     'CUDA x86' if HAS_CUDA else 'CPU')

GGUF_PATH = OUT_DIR / 'qwen3_1b7_q4km.gguf'
BENCH_BIN = LLAMA_DIR / 'build/bin/llama-bench'
CLI_BIN   = LLAMA_DIR / 'build/bin/llama-cli'

# Find rival GGUF
RIVAL_GGUF = None
for pat in ['gemma*Q4_K_M*.gguf', 'gemma*.gguf']:
    found = list(OUT_DIR.glob(pat))
    if found:
        RIVAL_GGUF = found[0]
        break

assert GGUF_PATH.is_file(),  'Run Cell 5 first'
assert RIVAL_GGUF and RIVAL_GGUF.is_file(), 'Run Cell 6 first'


def _bench(gguf_path):
    """Run llama-bench CSV; fall back to llama-cli timing lines."""
    if BENCH_BIN.is_file():
        r = subprocess.run([
            str(BENCH_BIN),
            '-m', str(gguf_path),
            '--n-gpu-layers', str(N_GPU_LAYERS),
            '-p', '512', '-n', '128', '-r', '3',
            '--output', 'csv',
        ], capture_output=True, text=True, timeout=300)
        pp_ts = tg_ts = None
        for line in r.stdout.splitlines():
            parts = [x.strip() for x in line.split(',')]
            if len(parts) < 2:
                continue
            try:
                test_f = parts[-2].lower()
                ts_f   = float(parts[-1])
                if 'pp' in test_f:
                    pp_ts = ts_f
                elif 'tg' in test_f:
                    tg_ts = ts_f
            except (ValueError, IndexError):
                pass
        if tg_ts:
            return pp_ts, tg_ts

    # Fallback: llama-cli
    r = subprocess.run([
        str(CLI_BIN), '-m', str(gguf_path),
        '--n-gpu-layers', str(N_GPU_LAYERS),
        '--ctx-size', '2048', '--n-predict', '128',
        '--log-disable',
        '-p', 'Axiom governs AI through cryptographic event tokens and HMAC-signed capsules:',
    ], capture_output=True, text=True, timeout=300)
    pp_ts = tg_ts = None
    for line in (r.stdout + r.stderr).splitlines():
        m = re.search(r'([\d.]+)\s+tokens per second', line)
        if not m:
            continue
        ts = float(m.group(1))
        ll = line.lower()
        if 'prompt eval' in ll:
            pp_ts = ts
        elif 'eval time' in ll:
            tg_ts = ts
    return pp_ts, tg_ts


MODELS = [
    ('Qwen3-1.7B  SRD Q4_K_M', GGUF_PATH),
    ('Gemma3-1B   Q4_K_M (rival)', RIVAL_GGUF),
]

results = {}
print(f'Benchmarking on: {BACKEND_LABEL}  (n_gpu_layers={N_GPU_LAYERS})\n')
for label, gguf in MODELS:
    size_mb = gguf.stat().st_size / 1024**2
    print(f'  Running: {label}  ({size_mb:.0f} MB) ...')
    t0 = time.time()
    pp, tg = _bench(gguf)
    wall   = time.time() - t0
    results[label] = {'pp': pp, 'tg': tg, 'mb': size_mb, 'wall_s': wall}
    print(f'    PP={pp:.1f if pp else "?"}  TG={tg:.1f if tg else "?"}  t/s   ({wall:.0f}s)')

# ── Comparison table ───────────────────────────────────────────────────────────
W = 72
print()
print('═' * W)
print(f'  INFERENCE BENCHMARK  [{BACKEND_LABEL}]  (PP=512 tokens, TG=128 tokens)')
print('═' * W)
print(f'  {"Model":<32}  {"Size MB":>8}  {"PP (t/s)":>10}  {"TG (t/s)":>10}')
print(f'  {"─"*32}  {"─"*8}  {"─"*10}  {"─"*10}')
for label, res in results.items():
    pp_s = f'{res["pp"]:.1f}' if res['pp'] else '—'
    tg_s = f'{res["tg"]:.1f}' if res['tg'] else '—'
    print(f'  {label:<32}  {res["mb"]:>8.0f}  {pp_s:>10}  {tg_s:>10}')
print('═' * W)

labels = list(results.keys())
if len(labels) == 2:
    our, rival = results[labels[0]], results[labels[1]]
    if our['tg'] and rival['tg']:
        d = (our['tg'] - rival['tg']) / rival['tg'] * 100
        print(f'\n  TG delta (Qwen3 SRD vs Gemma3 rival): {"+" if d>0 else ""}{d:.1f}%')
    if our['pp'] and rival['pp']:
        d = (our['pp'] - rival['pp']) / rival['pp'] * 100
        print(f'  PP delta (Qwen3 SRD vs Gemma3 rival): {"+" if d>0 else ""}{d:.1f}%')
    if our['mb'] and rival['mb']:
        d = (our['mb'] - rival['mb']) / rival['mb'] * 100
        print(f'  Size delta (Qwen3 SRD vs Gemma3 rival): {"+" if d>0 else ""}{d:.1f}%')

print('\n✓ Cell 8 complete — proceed to Cell 9')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Summary + download
# ══════════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

try:
    OUT_DIR
except NameError:
    import platform as _p
    _s = _p.system()
    WORKSPACE = Path('/content') if (Path('/content').is_dir() and _s != 'Darwin') else (Path.home()/'axiom_workspace')
    OUT_DIR   = WORKSPACE / 'qwen3_srd'

AXM_PATH     = OUT_DIR / 'qwen3_1b7_srd.axm'
GGUF_PATH    = OUT_DIR / 'qwen3_1b7_q4km.gguf'
SIDECAR_PATH = OUT_DIR / 'qwen3_1b7_q4km.axiom_meta.json'
STATS_PATH   = OUT_DIR / 'qwen3_1b7_pack_stats.json'

def _mb(p): return f'{p.stat().st_size/1024**2:.0f} MB' if p.is_file() else 'missing'

print('═' * 64)
print('  QWEN3-1.7B SRD · ARM · MET — OUTPUT SUMMARY')
print('═' * 64)
print()
print(f'  {"File":<42}  {"Size":>10}')
print(f'  {"─"*42}  {"─"*10}')
for label, path in [
    ('qwen3_1b7_srd.axm         (container)',  AXM_PATH),
    ('qwen3_1b7_q4km.gguf       (inference)',  GGUF_PATH),
    ('qwen3_1b7_q4km.axiom_meta.json  (MET)', SIDECAR_PATH),
    ('qwen3_1b7_pack_stats.json (stats)',      STATS_PATH),
]:
    print(f'  {label:<42}  {_mb(path):>10}')

if STATS_PATH.is_file():
    s = json.loads(STATS_PATH.read_text())
    print()
    print(f'  Fingerprint : {s.get("fingerprint", "?")}')
    print(f'  bpw         : {s.get("bpw_theoretical", "?")}')
    print(f'  Proofs      : {s.get("proofs", "?")}')

if SIDECAR_PATH.is_file():
    meta = json.loads(SIDECAR_PATH.read_text())
    print()
    print(f'  MET budget  : embed={meta["embedding_slot"]["mb"]:.0f} MB (F16, pinned)')
    for slot, info in meta['transformer_chunks'].items():
        print(f'              + {slot:<12} layers {info["layers"]}  {info["mb"]} MB Q4_K_M')
    print(f'  Peak HARM   : {meta["peak_harm_mb"]:.0f} MB total')

print()
print('═' * 64)

# ── Download (Colab) or show copy commands ────────────────────────────────────
try:
    from google.colab import files as _colab_files
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print('\nDownloading output files ...')
    for path in [AXM_PATH, GGUF_PATH, SIDECAR_PATH]:
        if path.is_file():
            print(f'  {path.name} ...')
            _colab_files.download(str(path))
    print('✓ Downloads triggered')
else:
    print(f'\nOutput directory: {OUT_DIR}')
    print()
    print('Deploy to Android device (via adb):')
    print(f'  adb push {GGUF_PATH.name} /storage/emulated/0/models/')
    print(f'  adb push {SIDECAR_PATH.name} /storage/emulated/0/models/')
    print()
    print('Run inference (llama.cpp):')
    gpu_layers = '99' if 'ARM NEON' not in str(globals().get('BACKEND_LABEL','')) else '0'
    print(f'  ./llama-cli -m {GGUF_PATH.name} --n-gpu-layers {gpu_layers} \\')
    print( '    -p "Axiom governs AI through cryptographic event tokens."')
    print()
    print('Verify container at any time:')
    print(f'  AXIOM_MASTER_KEY=<your-key> python3 axm_cli.py verify {AXM_PATH.name}')